# 03 — Volume Drivers (RQ1)

**Research Question:** What is driving hospitalization volume?

Kidney stone admissions in SP have grown significantly since 2016. We decompose this growth
to understand whether it reflects a true increase in disease incidence, changes in procedure
coding, or financial incentives favoring inpatient admission.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from shared import (
    load_kidney, setup_plot_style, save_plot, save_metrics,
    CATEGORY_MAP, PROC_NAMES, NEW_PROC, RAW_SIA_DIR,
)

setup_plot_style()
kidney = load_kidney()
k16 = kidney[kidney["year"] >= 2016]
print(f"Loaded {len(kidney):,} admissions ({len(k16):,} from 2016+)")
print(f"Year range: {int(kidney['year'].min())}–{int(kidney['year'].max())}")
print(f"Procedure codes: {kidney['PROC_REA'].nunique()}")
print(f"Procedure categories: {kidney['proc_category'].nunique()}")

Loaded 206,500 admissions (205,860 from 2016+)
Year range: 2015–2025
Procedure codes: 193
Procedure categories: 7


## 1. Procedure Taxonomy

Kidney stone admissions use ~15 SIGTAP procedure codes in six functional categories.

In [2]:
cat_summary = kidney.groupby("proc_category").agg(
    count=pd.NamedAgg(column="proc_category", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
).sort_values("count", ascending=False)
cat_summary["pct"] = cat_summary["count"] / len(kidney) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cat_summary["pct"].plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Share of Admissions by Category")
axes[0].set_xlabel("%")
axes[0].invert_yaxis()

cat_summary["mean_los"].plot.barh(ax=axes[1], color="#D97706")
axes[1].set_title("Mean LOS by Category")
axes[1].set_xlabel("Days")
axes[1].invert_yaxis()

cat_summary["mortality"].mul(100).plot.barh(ax=axes[2], color="#DC2626")
axes[2].set_title("Mortality Rate by Category")
axes[2].set_xlabel("%")
axes[2].invert_yaxis()

fig.suptitle("Procedure Categories", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "procedure_categories", prefix="03")
print(cat_summary.to_string())

  Saved: 03_procedure_categories.png
                count  mean_los    mean_cost  mortality        pct
proc_category                                                     
SURGICAL        88681  2.614777   983.143406   0.004601  42.944794
DIAGNOSTIC      41487  2.687734   368.743554   0.001904  20.090557
CLINICAL_MGMT   23275  2.388786  1508.381465   0.003308  11.271186
SURGICAL_MGMT   20597  2.247852  1243.601968   0.002088   9.974334
INTERVENTIONAL  20113  2.129916   976.516236   0.001740   9.739952
OBSERVATION      8818  0.580517   134.576585   0.001021   4.270218
OTHER            3529  4.030037  1073.827050   0.017852   1.708959


## 2. Procedure Adoption Timeline

When did ureteroscopy (0409010596) appear? How did it change the volume composition?

In [3]:
yearly_cat = kidney.groupby(["year", "proc_category"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
yearly_cat.plot.bar(stacked=True, ax=ax, width=0.8)
ax.set_title("Admission Volume by Procedure Category per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Admissions")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
save_plot(fig, "growth_by_category", prefix="03")

  Saved: 03_growth_by_category.png


In [4]:
# Ureteroscopy adoption timeline
yearly_new = kidney.groupby("year")["has_new_proc"].agg(["sum", "mean"]).rename(
    columns={"sum": "ureteroscopy_count", "mean": "ureteroscopy_share"}
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(yearly_new.index, yearly_new["ureteroscopy_count"], color="#7C3AED")
axes[0].set_title("Ureteroscopy Procedures per Year")
axes[0].set_ylabel("Count")

axes[1].plot(yearly_new.index, yearly_new["ureteroscopy_share"] * 100, "o-", color="#7C3AED")
axes[1].set_title("Ureteroscopy Share of All N20 Admissions")
axes[1].set_ylabel("%")

for ax in axes:
    ax.set_xlabel("Year")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Ureteroscopy (modern procedure) Adoption", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "ureteroscopy_adoption", prefix="03")

  Saved: 03_ureteroscopy_adoption.png


## 3. New vs Legacy Procedure Contribution

How much of the volume growth comes from new procedures (ureteroscopy) vs. legacy?

In [5]:
yearly_total = kidney.groupby("year").size().rename("total")
yearly_legacy = kidney[kidney["has_new_proc"] == 0].groupby("year").size().rename("legacy")
yearly_new_proc = kidney[kidney["has_new_proc"] == 1].groupby("year").size().rename("ureteroscopy")

decomp = pd.concat([yearly_total, yearly_legacy, yearly_new_proc], axis=1).fillna(0)

# Compute growth attribution
base_year = decomp.index.min()
base_total = decomp.loc[base_year, "total"]
decomp["total_growth"] = decomp["total"] - base_total
decomp["legacy_growth"] = decomp["legacy"] - decomp.loc[base_year, "legacy"]
decomp["new_proc_growth"] = decomp["ureteroscopy"] - decomp.loc[base_year, "ureteroscopy"]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(decomp.index, decomp["legacy_growth"], label="Legacy procedures", color="#6B7280")
ax.bar(decomp.index, decomp["new_proc_growth"], bottom=decomp["legacy_growth"],
       label="Ureteroscopy (new)", color="#7C3AED")
ax.set_title(f"Volume Growth Decomposition (relative to {base_year})")
ax.set_xlabel("Year")
ax.set_ylabel("Change in admissions")
ax.legend()
ax.tick_params(axis="x", rotation=45)
ax.axhline(0, color="black", linewidth=0.5)
fig.tight_layout()
save_plot(fig, "growth_decomposition", prefix="03")

  Saved: 03_growth_decomposition.png


## 3b. Statistical Growth Analysis — Per-Category CAGR and Trend Tests

For each procedure category, compute CAGR (2016–2025) and Kendall tau trend test.
This separates which categories are genuinely growing vs. stable.

In [6]:
yearly_cat_16 = k16.groupby(["year", "proc_category"]).size().unstack(fill_value=0)

cat_growth = []
for cat in yearly_cat_16.columns:
    series = yearly_cat_16[cat]
    first, last = series.iloc[0], series.iloc[-1]
    n_yrs = int(series.index[-1] - series.index[0])
    cagr = ((last / max(first, 1)) ** (1 / n_yrs) - 1) * 100 if n_yrs > 0 else 0
    tau, p = stats.kendalltau(series.index, series.values)
    cat_growth.append({
        "category": cat,
        "vol_2016": int(first),
        "vol_2025": int(last),
        "cagr_pct": round(cagr, 1),
        "kendall_tau": round(tau, 3),
        "p_value": p,
        "significant": p < 0.05,
    })

cat_growth_df = pd.DataFrame(cat_growth).sort_values("cagr_pct", ascending=False)
print("=== PER-CATEGORY GROWTH (2016–2025) ===")
print(cat_growth_df.to_string(index=False))

# Top individual procedures by absolute growth
proc_yearly = k16.groupby(["year", "proc_name"]).size().unstack(fill_value=0)
proc_growth = []
for proc in proc_yearly.columns:
    s = proc_yearly[proc]
    first, last = s.iloc[0], s.iloc[-1]
    abs_growth = int(last - first)
    tau, p = stats.kendalltau(s.index, s.values)
    proc_growth.append({
        "procedure": proc,
        "vol_2016": int(first),
        "vol_2025": int(last),
        "abs_growth": abs_growth,
        "kendall_tau": round(tau, 3),
        "p_value": p,
    })

proc_growth_df = pd.DataFrame(proc_growth).sort_values("abs_growth", ascending=False)
print("\n=== TOP PROCEDURES BY ABSOLUTE GROWTH (2016–2025) ===")
print(proc_growth_df.head(10).to_string(index=False))

# Growth attribution: what fraction of total growth came from ureteroscopy?
total_growth = int(yearly_cat_16.sum(axis=1).iloc[-1] - yearly_cat_16.sum(axis=1).iloc[0])
uretero_growth = int(yearly_cat_16.sum(axis=1).iloc[-1] - yearly_cat_16.sum(axis=1).iloc[0])
uretero_2016 = k16[(k16["year"]==2016) & (k16["has_new_proc"]==1)].shape[0]
uretero_2025 = k16[(k16["year"]==2025) & (k16["has_new_proc"]==1)].shape[0]
uretero_contribution = uretero_2025 - uretero_2016
legacy_contribution = total_growth - uretero_contribution
print(f"\nTotal volume growth 2016→2025: {total_growth:+,}")
print(f"  Ureteroscopy contribution: {uretero_contribution:+,} ({uretero_contribution/max(total_growth,1)*100:.1f}%)")
print(f"  Legacy procedures: {legacy_contribution:+,} ({legacy_contribution/max(total_growth,1)*100:.1f}%)")

=== PER-CATEGORY GROWTH (2016–2025) ===
      category  vol_2016  vol_2025  cagr_pct  kendall_tau  p_value  significant
      SURGICAL      4349     18013      17.1        0.956 0.000006         True
 CLINICAL_MGMT      1591      3528       9.3        0.867 0.000115         True
 SURGICAL_MGMT      1212      2489       8.3        0.644 0.009148         True
   OBSERVATION       678      1095       5.5        0.733 0.002213         True
         OTHER       282       439       5.0        0.733 0.002213         True
    DIAGNOSTIC      3853      4913       2.7        0.422 0.108313        False
INTERVENTIONAL      2269       885      -9.9       -0.511 0.046623         True



=== TOP PROCEDURES BY ABSOLUTE GROWTH (2016–2025) ===
                     procedure  vol_2016  vol_2025  abs_growth  kendall_tau  p_value
         Ureteroscopy (modern)         0     10907       10907        0.931 0.000299
         Open Ureterolithotomy      2749      5342        2593        0.956 0.000006
           Clinical Management      1591      3528        1937        0.867 0.000115
           Surgical Management      1212      2489        1277        0.644 0.009148
Diagnostic Imaging (Urography)      3805      4809        1004        0.422 0.108313
              ESWL Lithotripsy       149       738         589        0.156 0.600654
      Percutaneous Nephrostomy        84       369         285        0.822 0.000358
                ER Observation       436       702         266        0.822 0.000358
         Clinical Care (short)       242       393         151        0.689 0.004687
            Kidney Exploration        37        96          59        0.614 0.014912

Total vol

## 3c. Incidence vs Capture — Can We Tell?

If volume growth is a true epidemic, all procedure categories should grow proportionally.
If it's a capture/coding shift, newer or better-reimbursed procedures should grow faster.

In [7]:
# Normalized growth (2016=100) by category
norm = yearly_cat_16.div(yearly_cat_16.iloc[0]).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cat_colors = {"SURGICAL": "#2563EB", "DIAGNOSTIC": "#DC2626", "CLINICAL_MGMT": "#D97706",
              "INTERVENTIONAL": "#059669", "SURGICAL_MGMT": "#7C3AED",
              "OBSERVATION": "#6B7280", "OTHER": "#9CA3AF"}

for cat in norm.columns:
    axes[0].plot(norm.index, norm[cat], "o-", label=cat, color=cat_colors.get(cat, "#333"),
                linewidth=2 if cat in ["SURGICAL", "DIAGNOSTIC"] else 1)
axes[0].axhline(100, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title("Normalized Volume (2016=100) by Category")
axes[0].set_ylabel("Index (2016=100)")
axes[0].set_xlabel("Year")
axes[0].legend(fontsize=8)

# CAGR bar chart
cg = cat_growth_df.copy()
colors = [cat_colors.get(c, "#333") for c in cg["category"]]
axes[1].barh(range(len(cg)), cg["cagr_pct"], color=colors)
axes[1].set_yticks(range(len(cg)))
axes[1].set_yticklabels(cg["category"])
axes[1].set_title("CAGR by Category (2016–2025)")
axes[1].set_xlabel("CAGR (%)")
axes[1].invert_yaxis()
for i, (v, sig) in enumerate(zip(cg["cagr_pct"], cg["significant"])):
    marker = "***" if sig else "ns"
    axes[1].text(v + 0.3, i, f"{v:.1f}% {marker}", va="center", fontsize=8)

fig.suptitle("Growth Patterns — Incidence vs Capture Shift?", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "incidence_vs_capture", prefix="03")

# Test: is diagnostic growth faster than surgical?
diag_series = yearly_cat_16.get("DIAGNOSTIC", pd.Series([0]))
surg_series = yearly_cat_16.get("SURGICAL", pd.Series([0]))
if len(diag_series) == len(surg_series):
    diag_growth_rates = diag_series.pct_change().dropna()
    surg_growth_rates = surg_series.pct_change().dropna()
    u, p = stats.mannwhitneyu(diag_growth_rates, surg_growth_rates, alternative="two-sided")
    print(f"Diagnostic vs Surgical YoY growth rates: Mann-Whitney U p = {p:.4f}")
    print(f"  Diagnostic mean YoY growth: {diag_growth_rates.mean()*100:.1f}%")
    print(f"  Surgical mean YoY growth: {surg_growth_rates.mean()*100:.1f}%")

  Saved: 03_incidence_vs_capture.png
Diagnostic vs Surgical YoY growth rates: Mann-Whitney U p = 0.1853
  Diagnostic mean YoY growth: 3.2%
  Surgical mean YoY growth: 18.4%


## 4. SIA Comparison: Inpatient vs Outpatient Billing

Are the same diagnostic procedure codes billed in outpatient (SIA)? If so, admitting patients
for diagnostics that could be done in ambulatory settings inflates inpatient volume.

In [8]:
# Load SIA data if available
sia_files = sorted(RAW_SIA_DIR.glob("PASP*.parquet"))
print(f"Found {len(sia_files)} SIA files")

# N20 procedure codes used in inpatient
inpatient_procs = kidney["PROC_REA"].unique()
print(f"Unique inpatient procedure codes for N20: {len(inpatient_procs)}")

if sia_files:
    # Sample a few SIA files to check for matching procedure codes
    sia_sample_files = sia_files[-6:]  # last 6 months
    sia_matches = []
    for f in sia_sample_files:
        try:
            sia = pd.read_parquet(f, columns=["PA_PROC_ID", "PA_VALAPR"])
            sia["PA_PROC_ID"] = sia["PA_PROC_ID"].astype(str).str.strip()
            matched = sia[sia["PA_PROC_ID"].isin(inpatient_procs)]
            if not matched.empty:
                sia_matches.append(matched)
        except Exception:
            pass

    if sia_matches:
        sia_matched = pd.concat(sia_matches, ignore_index=True)
        sia_matched["PA_VALAPR"] = pd.to_numeric(sia_matched["PA_VALAPR"], errors="coerce")

        sia_avg = sia_matched.groupby("PA_PROC_ID")["PA_VALAPR"].agg(["mean", "count"]).rename(
            columns={"mean": "sia_avg_cost", "count": "sia_count"}
        )
        sih_avg = kidney.groupby("PROC_REA")["VAL_TOT"].agg(["mean", "count"]).rename(
            columns={"mean": "sih_avg_cost", "count": "sih_count"}
        )

        comparison = sia_avg.join(sih_avg, how="inner")
        comparison["admission_premium"] = comparison["sih_avg_cost"] / comparison["sia_avg_cost"]
        print(f"\nProcedures found in both SIH and SIA: {len(comparison)}")
        print(comparison.sort_values("admission_premium", ascending=False).to_string())
    else:
        print("No matching procedures found in SIA outpatient data.")
        comparison = pd.DataFrame()
else:
    print("SIA data not available. Skipping outpatient comparison.")
    comparison = pd.DataFrame()

Found 121 SIA files
Unique inpatient procedure codes for N20: 193



Procedures found in both SIH and SIA: 14
            sia_avg_cost  sia_count  sih_avg_cost  sih_count  admission_premium
0305020056      0.000000          5   1618.978276         29                inf
0415040035      1.057059        204    963.790000          2         911.765721
0409010294     87.780000          1   2029.612383        554          23.121581
0409010090     32.680000        173    718.499130         23          21.985898
0409040010     12.970000          9    211.060000          1          16.272938
0409020079     32.680000          5    365.934286          7          11.197500
0409040215     34.100000          6    256.970000          3           7.535777
0409020176     51.150000          2    382.233824         34           7.472802
0409010170    133.356522         69    734.879795      40973           5.510640
0406020620     20.740000        140     79.052222          9           3.811583
0409040240    538.897350       1404    611.830000          1           1.13533

## 5. Diagnostic Admissions — A Volume Inflator?

How many admissions are purely diagnostic (imaging) that could potentially be outpatient?

In [9]:
diag_yearly = kidney[kidney["proc_category"] == "DIAGNOSTIC"].groupby("year").size()
total_yearly = kidney.groupby("year").size()
diag_pct = (diag_yearly / total_yearly * 100).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(diag_yearly.index, diag_yearly.values, color="#DC2626")
axes[0].set_title("Diagnostic Admissions per Year")
axes[0].set_ylabel("Count")

axes[1].plot(diag_pct.index, diag_pct.values, "o-", color="#DC2626")
axes[1].set_title("Diagnostic Admissions as % of Total")
axes[1].set_ylabel("%")

for ax in axes:
    ax.set_xlabel("Year")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Diagnostic Admissions Trend", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "diagnostic_volume", prefix="03")

total_diag = len(kidney[kidney["proc_category"] == "DIAGNOSTIC"])
print(f"Total diagnostic admissions: {total_diag:,} ({total_diag/len(kidney)*100:.1f}%)")

  Saved: 03_diagnostic_volume.png
Total diagnostic admissions: 41,487 (20.1%)


## Summary Metrics

In [10]:
latest_year = int(kidney["year"].max())

# Per-year volumes for 2016 and 2025
vol_2016 = len(k16[k16["year"] == 2016])
vol_2025 = len(k16[k16["year"] == 2025])
n_yrs = 2025 - 2016
cagr_total = ((vol_2025 / max(vol_2016, 1)) ** (1 / n_yrs) - 1) * 100

# Overall trend test
yearly_total_16 = k16.groupby("year").size()
tau_total, p_total = stats.kendalltau(yearly_total_16.index, yearly_total_16.values)

# Fastest-growing category
fastest = cat_growth_df.iloc[0]

metrics = {
    "procedure_categories": len(cat_summary),
    "surgical_pct": round(float(cat_summary.loc["SURGICAL", "pct"]), 1) if "SURGICAL" in cat_summary.index else 0,
    "diagnostic_pct": round(float(cat_summary.loc["DIAGNOSTIC", "pct"]), 1) if "DIAGNOSTIC" in cat_summary.index else 0,
    "ureteroscopy_total": int(kidney["has_new_proc"].sum()),
    "ureteroscopy_2025_share_pct": round(float(
        kidney[kidney["year"] == latest_year]["has_new_proc"].mean() * 100
    ), 1),
    "vol_2016": vol_2016,
    "vol_2025": vol_2025,
    "cagr_2016_2025_pct": round(cagr_total, 1),
    "volume_trend_tau": round(float(tau_total), 3),
    "volume_trend_p": float(p_total),
    "fastest_growing_category": str(fastest["category"]),
    "fastest_cagr_pct": float(fastest["cagr_pct"]),
    "ureteroscopy_contribution_to_growth_pct": round(uretero_contribution / max(total_growth, 1) * 100, 1),
    "legacy_contribution_to_growth_pct": round(legacy_contribution / max(total_growth, 1) * 100, 1),
    "sia_procedures_matched": len(comparison) if not comparison.empty else 0,
    "total_diagnostic_admissions": int(len(kidney[kidney["proc_category"] == "DIAGNOSTIC"])),
}
save_metrics(metrics, "03_volume_drivers")

print("\n=== VOLUME DRIVERS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 03_volume_drivers.json

=== VOLUME DRIVERS ===
  procedure_categories: 7
  surgical_pct: 42.9
  diagnostic_pct: 20.1
  ureteroscopy_total: 34036
  ureteroscopy_2025_share_pct: 34.8
  vol_2016: 14234
  vol_2025: 31362
  cagr_2016_2025_pct: 9.2
  volume_trend_tau: 0.911
  volume_trend_p: 2.9761904761904762e-05
  fastest_growing_category: SURGICAL
  fastest_cagr_pct: 17.1
  ureteroscopy_contribution_to_growth_pct: 63.7
  legacy_contribution_to_growth_pct: 36.3
  sia_procedures_matched: 14
  total_diagnostic_admissions: 41487
